# Packages import

In [21]:
import os
import yaml
import requests
import pandas as pd
import re
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# Apollo Scraper

In [22]:
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2"

In [23]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['username']
password = config['credentials']['password']
auth = HTTPBasicAuth(username, password)

In [24]:
response = requests.get(url, auth=auth)
response.encoding = "UTF-8"
print(response.status_code)

200


In [25]:
page_dom = BeautifulSoup(response.text, 'html.parser')

In [26]:
group = page_dom.select_one("div.grupa").get_text()
print(group)

ZICSS1-1212


In [27]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(str(classes_tag.prettify()))
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [28]:
classes = classes.loc[classes['Typ'].isin(["ćwiczenia","wykład","egzamin"])]

In [29]:
classes[['Day', 'Start Time', 'hyphen', 'End Time', 'Duration']] = classes['Dzień, godzina'].str.split(' ',expand=True)

In [30]:
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [31]:
classes = classes.drop(['Dzień, godzina', 'hyphen'], axis=1)

In [32]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.)*",
    r"\1",
    regex=True
)

In [33]:
if not os.path.exists("./schedules"):
    os.mkdir("./schedules")

In [34]:
classes.to_csv(f"schedules/{group}.csv", encoding="UTF-8")

In [35]:
classes

,Termin,Przedmiot,Typ,Nauczyciel,Sala,Day,Start Time,End Time,Duration
1,2026-02-23,Computer Programming 2,ćwiczenia,dr Katarzyna Wójcik,"Paw.A 013 lab. Win10, Office21",Pn,15:45,18:15,3
4,2026-02-24,Probability and Statistics,wykład,prof. dr hab. Andrzej Sokołowski,Paw.F 008,Wt,09:45,11:15,2
6,2026-02-24,Discrete mathematics,wykład,dr Grzegorz Kosiorowski,Rakowicka 16 sala 11,Wt,15:00,16:30,2
7,2026-02-24,Business Law,wykład,dr Jacek Lachner,Rakowicka 16 sala 11,Wt,18:30,20:00,2
9,2026-02-26,Operating Systems and Computer Networks,wykład,prof. UEK dr hab. Joanna Wyrobek,Paw.F 008,Cz,08:00,09:30,2
...,...,...,...,...,...,...,...,...,...
292,2026-06-11,Operating Systems and Computer Networks,ćwiczenia,prof. UEK dr hab. Joanna Wyrobek,"Paw.A 301a lab.Win10, Office21",Cz,13:15,14:45,2
294,2026-06-11,Operating Systems and Computer Networks,ćwiczenia,prof. UEK dr hab. Joanna Wyrobek,"Paw.A 07 lab. Win7, Office21",Cz,16:45,18:15,2
296,2026-06-11,Information Systems,ćwiczenia,mgr inż. Justyna Olczak,"Paw.A 013 lab. Win10, Office21",Cz,18:30,19:15,1
297,2026-06-17,Discrete mathematics,egzamin,dr Grzegorz Kosiorowski,Paw.C Nowa Aula,Śr,10:30,13:00,3
